## Közös havi ingestion futtatás

Ez a cella a kiválasztott brókerek pipeline-jait futtatja havi bontásban.

Ha `START_MONTH` és `END_MONTH` értéke `None`, akkor a futás a `config/download_period.csv` szerinti időszakot használja.  
Ha megadjuk őket `YYYY-MM` formátumban, akkor csak az adott teljes, lezárt hónapok kerülnek feldolgozásra.

A futás hónaponként halad: az adott hónap kiválasztott Binance itemei után az adott hónap kiválasztott Dukascopy itemei futnak, majd jön a következő hónap.

A `BROKERS` paraméterrel szabályozható, mely brókerek fussanak:
- `None`: minden támogatott bróker
- `["binance"]`: csak Binance
- `["dukascopy"]`: csak Dukascopy
- `["binance", "dukascopy"]`: mindkettő

Az `ASSETS` paraméterrel szabályozható, mely assetek fussanak:
- `None`: minden, amit az adott bróker támogat a config szerint
- `["BTCUSD"]`: csak BTCUSD
- `["XAUUSD", "EURUSD"]`: csak a megadott assetek

Ha egy bróker nem támogat egy megadott assetet, arra nem jön létre futtatandó item.


In [ ]:
from pipelines.monthly_ingestion_runner import run_monthly_ingestion_from_config


# Ha mindkettő None, akkor a config/download_period.csv szerinti időszak fut.
# Ha megadod őket, csak a megadott teljes hónapok futnak.
START_MONTH = None
END_MONTH = None

# Példa explicit időszakra:
# START_MONTH = "2024-02"
# END_MONTH = "2024-03"

# Futtatandó brókerek.
# None = minden támogatott bróker
# ["binance"] = csak Binance
# ["dukascopy"] = csak Dukascopy
# ["binance", "dukascopy"] = mindkettő
BROKERS = ["binance", "dukascopy"]

# Futtatandó assetek.
# None = minden configban támogatott asset
# ["BTCUSD"] = csak BTCUSD
ASSETS = None

results_df = run_monthly_ingestion_from_config(
    start_month=START_MONTH,
    end_month=END_MONTH,
    brokers=BROKERS,
    assets=ASSETS,
    interval="1m",
    dukascopy_print_progress=True,
)

results_df



=== 2024-01 ===
Binance indul...
Dukascopy indul...


## Binance havi letöltés

Ez a cella a Binance pipeline-t futtatja a megadott teljes hónapokra.

Ha nincs megadva `start_month` és `end_month`, akkor a `config/download_period.csv` szerinti időszak futna.  
Ha megadjuk őket, akkor csak az adott lezárt hónapok kerülnek feldolgozásra.

Fontos: csak teljes, múltbeli hónap tölthető le. Az aktuális hónap nem engedélyezett!


In [3]:
from pipelines.binance_runner import run_binance_from_config

import pandas as pd

# Letöltendő teljes hónapok:
# Kezdő hónap: 2024-01
# Záró hónap: 2024-03
#
# A pipeline:
# - Binance BTCUSDT 1 perces havi ZIP fájlokat használ
# - raw CSV-t ment/feltölt
# - bronze OHLCV parquet fájlt készít/feltölt
# - manifestet és _SUCCESS markert ír
# - meglévő _SUCCESS esetén skipeli az adott hónapot

results = run_binance_from_config(
    start_month="2024-01",
    end_month="2024-03",
    interval="1m",
)

pd.DataFrame([result.__dict__ for result in results])

,status,broker,asset,broker_symbol,year,month,message
0,skipped,binance,BTCUSD,BTCUSDT,2024,1,_SUCCESS already exists in Azure.
1,skipped,binance,BTCUSD,BTCUSDT,2024,2,_SUCCESS already exists in Azure.
2,skipped,binance,BTCUSD,BTCUSDT,2024,3,_SUCCESS already exists in Azure.


## Dukascopy havi letöltés

Ez a cella a Dukascopy pipeline-t futtatja a megadott teljes hónapokra.

Ha nincs megadva `start_month` és `end_month`, akkor a `config/download_period.csv` szerinti időszak futna.  
Ha megadjuk őket, akkor csak az adott lezárt hónapok kerülnek feldolgozásra.

Fontos: csak teljes, múltbeli hónap tölthető le.  
Az aktuális hónap nem engedélyezett.

A Dukascopy pipeline órás `.bi5` tick fájlokat tölt le, ezekből raw tick parquetet, majd bronze OHLCV parquetet készít. Emiatt egy teljes hónap több assetre hosszabb ideig futhat.


In [4]:
from pipelines.dukascopy_runner import run_dukascopy_from_config

import pandas as pd

# Letöltendő teljes hónapok:
# Kezdő hónap: 2024-01
# Záró hónap: 2024-01
#
# A pipeline:
# - Dukascopy órás .bi5 tick fájlokat használ
# - raw tick parquet fájlt készít/feltölt
# - bronze OHLCV parquet fájlt készít/feltölt
# - manifestet és _SUCCESS markert ír
# - meglévő _SUCCESS esetén skipeli az adott hónap/asset párt
#
# Fejlesztési próba:
# start_index=5, limit=1 csak a 2024-01 BTCUSD itemet futtatja.
#
# 2024-01 Dukascopy itemek:
# 0 = XAUUSD
# 1 = XAGUSD
# 2 = EURUSD
# 3 = US500
# 4 = DAX
# 5 = BTCUSD

results = run_dukascopy_from_config(
    start_month="2024-01",
    end_month="2024-01",
    interval="1m",
    start_index=5,
    limit=1,
    timeout_sec=5,
    max_attempts=2,
    retry_sleep_sec=1,
    print_progress=True,
)

pd.DataFrame([result.__dict__ for result in results])


,status,broker,asset,broker_symbol,year,month,message,errors
0,skipped,dukascopy,BTCUSD,BTCUSD,2024,1,_SUCCESS already exists in Azure.,[]
